In [ ]:
!pip install -q mlxtend

In [ ]:
# Import necessary data manipulation and association rule mining libraries
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
# Suppress deprecation warnings for cleaner output
warnings.filterwarnings("ignore")
print("Environment and mlxtend successfully configured!")

Environment and mlxtend successfully configured!


#Step 2: Generate Synthetic Retail POS Basket Logs

In [ ]:
np.random.seed(42)
n_baskets = 8000
# Define retail catalog and realistic co-occurrence bundles
bundles = [
 ['Artisan Bread', 'Organic Butter', 'Whole Milk'],
 ['Artisan Bread', 'Strawberry Jam', 'Whole Milk'],
 ['Arabica Coffee', 'Raw Sugar', 'Oat Creamer'],
 ['Arabica Coffee', 'Croissant'],
 ['Wireless Mouse', 'USB-C Hub', 'Laptop Sleeve'],
 ['Pasta Penne', 'Marinara Sauce', 'Parmesan Cheese'],
 ['Tortilla Chips', 'Salsa Dip', 'Guacamole'],
 ['Shampoo', 'Conditioner', 'Body Wash']
]
standalone_items = [
 'Sparkling Water', 'Dark Chocolate', 'Bananas', 'Greek Yogurt',
 'Paper Towels', 'AA Batteries', 'Notebook', 'Olive Oil'
]
# Generate basket records
records = []
for basket_idx in range(1, n_baskets + 1):
    basket_id = f"BKT_{str(basket_idx).zfill(5)}"
    # Select 1 primary bundle and 0-2 random standalone items
    chosen_bundle = bundles[np.random.randint(0, len(bundles))]
    extra_count = np.random.choice([0, 1, 2], p=[0.4, 0.4, 0.2])
    extras = list(np.random.choice(standalone_items, size=extra_count, replace=False))

    # Combine items and remove accidental duplicates within the same basket
    basket_items = list(set(chosen_bundle + extras))
    for item in basket_items:
        records.append({'BasketID': basket_id, 'Product': item, 'Quantity': 1})
# Create DataFrame and export raw POS logs
pos_df = pd.DataFrame(records)
pos_df.to_csv('retail_pos_basket_logs.csv', index=False)
print(f"Dataset created: {len(pos_df)} line items across {pos_df['BasketID'].nunique()} unique baskets!")
pos_df.head(10)

Dataset created: 29457 line items across 8000 unique baskets!


,BasketID,Product,Quantity
0,BKT_00001,Guacamole,1
1,BKT_00001,Greek Yogurt,1
2,BKT_00001,Tortilla Chips,1
3,BKT_00001,Salsa Dip,1
4,BKT_00002,Guacamole,1
5,BKT_00002,Tortilla Chips,1
6,BKT_00002,Salsa Dip,1
7,BKT_00003,Olive Oil,1
8,BKT_00003,Body Wash,1
9,BKT_00003,Conditioner,1


In [ ]:
basket_matrix = (
 pos_df.groupby(['BasketID', 'Product'])['Quantity']
 .sum()
 .unstack()
 .fillna(0)
)
# Convert positive counts to boolean (True/False) for optimal mlxtend performance
basket_encoded = basket_matrix.map(lambda x: True if x > 0 else False)
print(f"Encoded Matrix Shape: {basket_encoded.shape[0]} baskets x {basket_encoded.shape[1]} unique products")
basket_encoded.head()


Encoded Matrix Shape: 8000 baskets x 28 unique products


Product,AA Batteries,Arabica Coffee,Artisan Bread,Bananas,Body Wash,Conditioner,Croissant,Dark Chocolate,Greek Yogurt,Guacamole,...,Pasta Penne,Raw Sugar,Salsa Dip,Shampoo,Sparkling Water,Strawberry Jam,Tortilla Chips,USB-C Hub,Whole Milk,Wireless Mouse
BasketID,,,,,,,,,,,,,,,,,,,,,
BKT_00001,False,False,False,False,False,False,False,False,True,True,...,False,False,True,False,False,False,True,False,False,False
BKT_00002,False,False,False,False,False,False,False,False,False,True,...,False,False,True,False,False,False,True,False,False,False
BKT_00003,False,False,False,True,True,True,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
BKT_00004,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,True
BKT_00005,False,False,False,False,False,False,False,True,False,False,...,True,False,False,False,False,False,False,False,False,False


In [ ]:
# Extract frequent itemsets with min_support = 3%
frequent_itemsets = apriori(basket_encoded, min_support=0.03, use_colnames=True)
# Add itemset length column for inspection
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
print(f"Total frequent itemsets discovered: {len(frequent_itemsets)}")
frequent_itemsets.sort_values(by='support', ascending=False).head(10)


Total frequent itemsets discovered: 56


,support,itemsets,length
1,0.250625,(Arabica Coffee),1
2,0.248250,(Artisan Bread),1
33,0.248250,"(Whole Milk, Artisan Bread)",2
26,0.248250,(Whole Milk),1
47,0.128875,"(Strawberry Jam, Whole Milk)",2
32,0.128875,"(Strawberry Jam, Artisan Bread)",2
23,0.128875,(Strawberry Jam),1
51,0.128875,"(Strawberry Jam, Whole Milk, Artisan Bread)",3
24,0.128750,(Tortilla Chips),1
37,0.128750,"(Guacamole, Salsa Dip)",2


In [ ]:
# Generate association rules using Lift as the primary evaluation metric
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.5)
# Filter for actionable high-confidence and high-support rules
strong_rules = rules[
 (rules['support'] >= 0.04) &
 (rules['confidence'] >= 0.40) &
 (rules['lift'] >= 1.5)
].copy()
# Convert frozensets into clean comma-separated strings for Power BI compatibility
strong_rules['Antecedent'] = strong_rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
strong_rules['Consequent'] = strong_rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
# Select and round key columns for export
export_cols = ['Antecedent', 'Consequent', 'support', 'confidence', 'lift', 'leverage', 'conviction']
final_rules_df = strong_rules[export_cols].round(4).sort_values(by='lift', ascending=False)
# Export to CSV for Power BI ingestion
final_rules_df.to_csv('apriori_association_rules.csv', index=False)
print(f"Exported {len(final_rules_df)} high-impact association rules to 'apriori_association_rules.csv'!")
final_rules_df.head(10)

Exported 84 high-impact association rules to 'apriori_association_rules.csv'!


,Antecedent,Consequent,support,confidence,lift,leverage,conviction
28,Pasta Penne,Marinara Sauce,0.1218,1.0,8.2136,0.1069,inf
29,Marinara Sauce,Pasta Penne,0.1218,1.0,8.2136,0.1069,inf
27,Marinara Sauce,Parmesan Cheese,0.1218,1.0,8.2136,0.1069,inf
26,Parmesan Cheese,Marinara Sauce,0.1218,1.0,8.2136,0.1069,inf
82,Marinara Sauce,"Parmesan Cheese, Pasta Penne",0.1218,1.0,8.2136,0.1069,inf
83,Pasta Penne,"Marinara Sauce, Parmesan Cheese",0.1218,1.0,8.2136,0.1069,inf
34,Parmesan Cheese,Pasta Penne,0.1218,1.0,8.2136,0.1069,inf
35,Pasta Penne,Parmesan Cheese,0.1218,1.0,8.2136,0.1069,inf
81,Parmesan Cheese,"Marinara Sauce, Pasta Penne",0.1218,1.0,8.2136,0.1069,inf
80,"Marinara Sauce, Pasta Penne",Parmesan Cheese,0.1218,1.0,8.2136,0.1069,inf
